In [1]:
%pip install tensorflow

In [2]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np
import re
#import matplotlib.pyplot as plt
import tensorflow as tf

# Import necessary layers for building the neural network
from tensorflow.keras.layers import  Dropout,  Activation
# Import the Sequential model API for creating a linear stack of layers
from tensorflow.keras.models import Sequential
# Import optimizers for updating the model's weights during training
from tensorflow.keras.optimizers import SGD, Adam
# Import callbacks for training behavior control
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from datetime import datetime
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



In [ ]:


# Use Keras tokenizer and pad_sequences via the already-imported tensorflow
Tokenizer = tf.keras.preprocessing.text.Tokenizer
pad_sequences = tf.keras.preprocessing.sequence.pad_sequences

# --- 1. CLEAN & TOKENIZE DATA ---
path_to_file ="..\\data\\shakespeare.txt"
text = open(path_to_file, 'rb').read().decode(encoding='utf-8').lower()

# Slice a portion of text to prevent running out of RAM/Memory
#text = "To be, or not to be, that is the question."  # Replace with file read later
text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation & lowercase
# print(f"Cleaned Text: {text}")
corpus = text.split('\n')[:1500] 

# 2. Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1  # Add 1 for the padding token

# 3. Create N-gram input sequences
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences so they have uniform lengths
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))


# Split data into predictors (X) and label targets (y)
X, y = input_sequences[:,:-1], input_sequences[:,-1]


# 4. Build the Bidirectional LSTM Network
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len-1),
    Bidirectional(LSTM(150, return_sequences=True)),
    LSTM(100),
    Dense(total_words, activation='softmax') # Multi-class activation
])

# Compile using Sparse Categorical Crossentropy to save memory over One-Hot formats
model.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)


# 6. Predict the Next Words Function
def generate_poetry(seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_idx = np.argmax(predicted_probs, axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_idx:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Example generation
# 5. Train the Model
model.fit(X, y, epochs=180, batch_size=64, verbose=1)
print(generate_poetry("to be or not to be", next_words=5))



Epoch 1/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.0310 - loss: 6.9170
Epoch 2/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - accuracy: 0.0338 - loss: 6.4521
Epoch 3/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 90ms/step - accuracy: 0.0390 - loss: 6.2976
Epoch 4/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 74ms/step - accuracy: 0.0445 - loss: 6.1861
Epoch 5/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 71ms/step - accuracy: 0.0501 - loss: 6.0902
Epoch 6/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.0541 - loss: 5.9897
Epoch 7/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.0579 - loss: 5.8975
Epoch 8/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.0606 - loss: 5.8111
Epoch 9/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.0665 - loss: 5.7303
Epoch 10/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - accuracy: 0.0637 - loss: 5.6508
Epoch 11/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 0.0688 - loss: 5.5761
Epoch 12/180
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 63ms/step